# 01 — Data Loading & Inventory (bulk CSV)

Goal: load the three GROW datasets from the FAOSTAT **bulk downloads** (no account),
confirm they melt to long format correctly, and inventory what's inside — areas,
items, elements, year coverage — before diving into analysis.

**Setup:** extract each bulk zip so the `_All_Data.csv` files sit in `data/raw/`
(or leave them in Downloads — the loader also checks `~/Downloads/<name>_All_Data/`).

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
from src.load import load_dataset, load_codes, GROW_DATASETS

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

## 1. Load all three (first run melts wide→long and caches to parquet; later runs are instant)

In [2]:
qcl = load_dataset("QCL")
qi  = load_dataset("QI")
qv  = load_dataset("QV")
for name, df in [("QCL", qcl), ("QI", qi), ("QV", qv)]:
    print(f"{name}: {df.shape[0]:,} rows, years {df['year'].min()}–{df['year'].max()}")

[cache] QCL_long.parquet
[cache] QI_long.parquet
[cache] QV_long.parquet
QCL: 4,114,755 rows, years 1961–2024
QI: 1,995,192 rows, years 1961–2024
QV: 3,392,180 rows, years 1961–2024


## 2. What do the columns look like?

In [3]:
qcl.head()

,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Unit,year,value,flag
0,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,ha,1961,0.0,A
1,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5510,Production,t,1961,0.0,A
2,2,'004,Afghanistan,515,'01341,Apples,5312,Area harvested,ha,1961,2220.0,E
3,2,'004,Afghanistan,515,'01341,Apples,5412,Yield,kg/ha,1961,6801.8,E
4,2,'004,Afghanistan,515,'01341,Apples,5510,Production,t,1961,15100.0,X


## 3. Inventory QCL — the richest dataset
What elements, how many items, how many areas vs aggregates?

In [4]:
print("Elements:")
print(qcl[["Element Code", "Element", "Unit"]].drop_duplicates().to_string(index=False))

Elements:
 Element Code                       Element    Unit
         5312                Area harvested      ha
         5510                    Production       t
         5412                         Yield   kg/ha
         5111                        Stocks      An
         5320 Producing Animals/Slaughtered      An
         5112                        Stocks 1000 An
         5413                         Yield   No/An
         5424          Yield/Carcass Weight    g/An
         5513                    Production 1000 No
         5313                        Laying 1000 An
         5417          Yield/Carcass Weight   kg/An
         5321 Producing Animals/Slaughtered 1000 An
         5318                  Milk Animals      An
         5114                        Stocks      No


In [5]:
items = qcl[["Item Code", "Item"]].drop_duplicates()
print(f"QCL items: {len(items)}")
items.head(30)

QCL items: 301


,Item Code,Item
0,221,"Almonds, in shell"
2,515,Apples
5,526,Apricots
8,1107,Asses
9,44,Barley
12,983,Butter and ghee of sheep milk
13,886,Butter of cow milk
14,1126,Camels
15,568,Cantaloupes and other melons
18,866,Cattle


In [6]:
areas = qcl[["Area Code", "Area"]].drop_duplicates()
agg = areas[pd.to_numeric(areas["Area Code"], errors="coerce") >= 5000]
print(f"Total areas: {len(areas)} | aggregates (code>=5000): {len(agg)}")
print("Aggregates include:", agg["Area"].head(15).tolist())

Total areas: 244 | aggregates (code>=5000): 34
Aggregates include: ['World', 'Africa', 'Eastern Africa', 'Middle Africa', 'Northern Africa', 'Southern Africa', 'Western Africa', 'Americas', 'Northern America', 'Central America', 'Caribbean', 'South America', 'Asia', 'Eastern Asia', 'Southern Asia']


## 4. Same quick inventory for QI and QV elements

In [7]:
print("QI elements:")
print(qi[["Element Code", "Element", "Unit"]].drop_duplicates().to_string(index=False))
print("\nQV elements:")
print(qv[["Element Code", "Element", "Unit"]].drop_duplicates().to_string(index=False))

QI elements:
 Element Code                                                    Element  Unit
          432            Gross Production Index Number (2014-2016 = 100)   NaN
          434 Gross per capita Production Index Number (2014-2016 = 100)   NaN

QV elements:
 Element Code                                                  Element      Unit
          152  Gross Production Value (constant 2014-2016 thousand I$) 1000 Int$
           55 Gross Production Value (constant 2014-2016 thousand SLC)  1000 SLC
           58 Gross Production Value (constant 2014-2016 thousand US$)  1000 USD
           56            Gross Production Value (current thousand SLC)  1000 SLC
           57            Gross Production Value (current thousand US$)  1000 USD


## 5. Peek at the shipped lookup tables (optional reference)
ItemCodes flags which items are aggregates; Elements confirms units.

In [8]:
try:
    item_codes = load_codes("QCL", "ItemCodes")
    print("ItemCodes sample:")
    print(item_codes.head())
except FileNotFoundError as e:
    print("(ItemCodes CSV not extracted — optional)", e)

(ItemCodes CSV not extracted — optional) Could not find Production_Crops_Livestock_E_ItemCodes.csv in /mnt/d/Sweta/Northeastern/Codes/python/wid-datathon-grow-eda/data/raw, /home/mary/Downloads/Production_Crops_Livestock_E_All_Data, /home/mary/Downloads. Extract the bulk zip into data/raw/ or pass search_dirs=[...].


**Checkpoint:** all three load and melt cleanly, coverage runs 1961→recent.
Note the exact Element Codes above — you'll filter on them in notebooks 02–04.
Move to `02_top_producers.py`.